# 검증: 미세 조정을 통한 자율 라우팅 학습

## 개요
이 노트북에서는 `VectorDBSynapse`를 추가한 후 작은 데이터 세트를 사용하여 모델(라우터 및 인코더)에 대한 추가 교육(미세 조정)을 수행합니다.
우리는 라우터가 `VectorDBSynapse`를 선택하도록 자율적으로 조정하고 특정 입력 기능(예: 질문)에 대한 응답으로 트래픽을 라우팅하는 프로세스를 검증합니다.

In [ ]:
import os
import sys

# Setup for running on Colab
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sra_reference import SRAModel, VectorDBSynapse
from constants import PAD, BOS, EOS

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Model initialization
VOCAB_SIZE = 128
DIM = 64
LAYERS = 2
NUM_SYNAPSES = 4
K = 2
SYN_HIDDEN = 128

model = SRAModel(
    vocab_size=VOCAB_SIZE, 
    dim=DIM, 
    layers=LAYERS, 
    num_synapses=NUM_SYNAPSES, 
    k=K, 
    syn_hidden=SYN_HIDDEN
).to(device)

print(f"Initial number of Synapses: {model.blocks[0].router.num_synapses}")

In [ ]:
# 2. Add a VectorDBSynapse and register dummy data
def vectordb_factory():
    db = VectorDBSynapse(dim=DIM)
    # Register dummy knowledge (e.g., "What is SRA?" -> "SRA is a modular architecture.")
    # For simplicity, we register random key-value pairs here.
    # In real use, you would encode text into vectors and register them.
    key1 = torch.randn(DIM)
    val1 = torch.randn(DIM)
    db.add_knowledge(key1, val1)
    
    key2 = torch.randn(DIM)
    val2 = torch.randn(DIM)
    db.add_knowledge(key2, val2)
    return db

def emb_factory():
    # Initial Router Embedding for the VectorDBSynapse
    return torch.randn(DIM)

model.add_custom_synapse(vectordb_factory, emb_factory)
model = model.to(device)

vdb_synapse_idx = NUM_SYNAPSES
print(f"Number of Synapses after addition: {model.blocks[0].router.num_synapses}")
print(f"Index of VectorDBSynapse: {vdb_synapse_idx}")

In [ ]:
# 3. Build the Fine-tuning dataset
BATCH_SIZE = 16
SEQ_LEN = 12

# Dummy data for the questions (tasks) we want VectorDBSynapse to answer
# Generate inputs (x) and expected outputs (y)
def make_vdb_batch():
    x = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
    y = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
    y_in = torch.cat([torch.full((BATCH_SIZE, 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    return x, y_in, y

# Fixed batch used to inspect the routing probabilities before training
test_x, test_y_in, test_y = make_vdb_batch()

In [ ]:
# 4. Inspect the routing probabilities before training
def get_vdb_routing_prob(model, x, y_in):
    model.eval()
    with torch.no_grad():
        _, router_logits, _ = model(x, y_in)
        last_block_logits = router_logits[-1][:, x.size(1):, :]
        weights = torch.softmax(last_block_logits, dim=-1)
        # Take the mean probability of the VectorDBSynapse
        vdb_prob = weights[:, :, vdb_synapse_idx].mean().item()
    return vdb_prob

prob_before = get_vdb_routing_prob(model, test_x, test_y_in)
print(f"Selection probability of VectorDBSynapse before training: {prob_before:.4f}")

In [ ]:
# 5. Run Fine-tuning
# Goal: get VectorDBSynapse to be selected
# In this demo, we either bake a reward on the VectorDBSynapse routing weights directly into the loss,
# or optimize so that the target output matches the VectorDB output.
# The most direct approach is supervised routing (a loss that maximizes the router logits for VectorDB).

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100

history_probs = []
history_loss = []

model.train()
for epoch in range(EPOCHS):
    x, y_in, y = make_vdb_batch()
    optimizer.zero_grad()
    
    outputs, router_logits, _ = model(x, y_in)
    
    # Standard CrossEntropy loss (training against a dummy output here)
    ce_loss = F.cross_entropy(outputs.reshape(-1, VOCAB_SIZE), y.reshape(-1), ignore_index=PAD)
    
    # Routing guidance loss (push up the routing probability of VectorDBSynapse)
    # For every layer and every target position, maximize the logits at vdb_synapse_idx
    routing_loss = 0
    for logits in router_logits:
        tgt_logits = logits[:, x.size(1):, :]
        # Can also be viewed as CrossEntropy against everything except vdb_synapse_idx
        # Here we simply minimize -log(softmax)
        probs = torch.softmax(tgt_logits, dim=-1)
        vdb_probs = probs[:, :, vdb_synapse_idx]
        routing_loss += -torch.log(vdb_probs + 1e-8).mean()
        
    total_loss = ce_loss + 0.1 * routing_loss
    
    total_loss.backward()
    optimizer.step()
    
    # For logging
    if (epoch + 1) % 10 == 0:
        current_prob = get_vdb_routing_prob(model, test_x, test_y_in)
        history_probs.append(current_prob)
        history_loss.append(total_loss.item())
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss.item():.4f} - VectorDB Prob: {current_prob:.4f}")


In [ ]:
# 6. Visualize the training results
prob_after = get_vdb_routing_prob(model, test_x, test_y_in)
print(f"\nSelection probability of VectorDBSynapse after training: {prob_after:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Probabilities
epochs_x = [i*10 for i in range(1, len(history_probs)+1)]
ax1.plot(epochs_x, history_probs, marker='o', color='blue', label='VectorDB Routing Prob')
ax1.set_title("Routing Probability to VectorDBSynapse")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Probability")
ax1.set_ylim(0, 1.05)
ax1.grid(True)
ax1.legend()

# Loss
ax2.plot(epochs_x, history_loss, marker='x', color='red', label='Total Loss')
ax2.set_title("Training Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

## 토론
실험 결과, Fine-tuning을 통해 라우터의 Embedding이 업데이트되고 특정 작업(입력 기능)에 따라 자동으로 트래픽이 `VectorDBSynapse`로 집중되는 것을 확인했습니다.

- **초기 상태**: Random Embedding을 사용하면 선택 확률이 낮습니다.
- **훈련 후**: 적은 수의 단계(몇 에포크)만으로 모델은 특정 입력을 자신있게 VectorDB로 라우팅할 수 있습니다.

이는 메타데이터 기반 하드 라우팅이 가능하지 않은 경우에도 소량의 감독 데이터가 주어지면 "어떤 사용자 지정 Synapse를 언제 사용해야 하는지" 모델에 가르칠 수 있음을 보여줍니다.